In [1]:
import xarray as xr
import pandas as pd
import altair as alt
import os
from vega_datasets import data
import ipywidgets as widgets
from IPython.display import display, clear_output

import altair as alt
alt.data_transformers.disable_max_rows()
alt.data_transformers.enable('default')

# Load dataset (Zarr or NetCDF)
def load_dataset():
    ds = xr.open_dataset(os.path.join("outputs", "soil_moisture_output.zarr"))
    time_values = ds["datetime"].values
    non_empty_times = []
    for i, _ in enumerate(time_values):
        sliced_data = ds['soil_moisture'].isel({"datetime": i})
        if not sliced_data.isnull().all():
            non_empty_times.append(i)
    return ds, non_empty_times

ds, time_values_inds = load_dataset()
variable = "soil_moisture"
time_coord = "datetime"
time_values = ds[time_coord].values

# Create slider widget
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_values_inds) - 1,
    step=1,
    description='Time Index:',
    continuous_update=False
)

# Output area
output = widgets.Output()

# Callback to update plot
def update_plot(change):
    output.clear_output(wait=True)
    index = change['new']
    selected_time = time_values[time_values_inds[index]]
    formatted_time = pd.to_datetime(selected_time).strftime("%Y-%m-%d")

    da = ds[variable].isel({time_coord: time_values_inds[index]})
    df = da.to_dataframe().reset_index().dropna()

    # Optional: print for debug
    print(df.dtypes)
    print(df.head())

    # Filter for valid values (customize threshold if needed)
    filtered_df = df[df[variable] >= 1]

    with output:
        print(f"Selected Time: {formatted_time} (Index: {index})")

        if filtered_df.empty:
            print("No valid data to plot for this time step.")
            return
                
        filtered_df = df[df[variable] >= 1].copy()

        # 🩺 Ensure all columns are float-compatible, no timezone issues
        filtered_df["longitude"] = filtered_df["longitude"].astype(float)
        filtered_df["latitude"] = filtered_df["latitude"].astype(float)
        filtered_df[variable] = filtered_df[variable].astype(float)

        # 🧼 Drop datetime if not used in plot — or convert safely
        if "datetime" in filtered_df.columns:
            filtered_df["datetime"] = pd.to_datetime(filtered_df["datetime"]).dt.strftime("%Y-%m-%d %H:%M:%S")

        states = alt.topo_feature(data.us_10m.url, feature='states')
        background = alt.Chart(states).mark_geoshape(
            fill='lightgray',
            stroke='white'
        ).project('albersUsa').properties(
            width=500,
            height=300
        )

        points = alt.Chart(filtered_df).mark_circle().encode(
            longitude='longitude:Q',
            latitude='latitude:Q',
            size=alt.value(10),
            color=alt.Color(f"{variable}:Q", scale=alt.Scale(scheme="viridis")),
            tooltip=["latitude", "longitude", variable],
            opacity=alt.value(0.5)
        )

        chart = background + points
        display(chart)

# Attach callback
slider.observe(update_plot, names='value')

# Display the UI
display(slider)
display(output)

# Trigger initial rendering
update_plot({'new': slider.value})

/Users/draghun1/miniforge3/envs/crest_all/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


IntSlider(value=0, continuous_update=False, description='Time Index:', max=3)

Output()

latitude                float64
longitude               float64
datetime         datetime64[ns]
soil_moisture           float64
dtype: object
     latitude  longitude            datetime  soil_moisture
870     25.25     -81.25 2018-05-21 04:00:00            0.0
871     25.25     -81.00 2018-05-21 04:00:00            0.0
872     25.25     -80.75 2018-05-21 04:00:00            0.0
873     25.25     -80.50 2018-05-21 04:00:00            0.0
874     25.25     -80.25 2018-05-21 04:00:00            0.0
